# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Frederic7/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



### Plain-language rule (Lane 2: Refresh / Content Opportunity Scoring)
> **Rule:** A page is prioritized for an editorial refresh if it has proven search visibility (`impressions_90d >= 500`) and has gone un-updated for at least 90 days (`days_since_last_update >= 90`). Pages meeting both thresholds receive an editorial priority score scaling their search volume with their staleness risk, ensuring high-demand pages decaying from neglect rise to the top of the queue.

- **Score formula:** `score = (days_since_last_update / 365.0) * log1p(impressions_90d)` for qualifying pages, `0.0` otherwise.
- **Reason code:** `stale_visible_page` (for qualifying pages) or `monitor_low_priority` (for non-qualifying pages).
- **Suggested action:** `refresh` (top queue) or `monitor`.

---

### Signal Check 1 (Flag-Linked): Staleness (`days_since_last_update`) behind FlyRank refresh flags
- **Hypothesis:** Pages that have not been updated recently suffer freshness decay in Google rankings and have a higher likelihood of declining in search performance (`is_declining_label == 1`).
- **Verdict:** **MIXED**
- **Reasoning:** In the data, decline rates rise steadily from fresh content (`0–30 days`: 51.1%) to stale content (`91–180 days`: 61.1%). However, for the oldest tier (`181+ days`), the observed decline rate drops back to 47.1% (survivorship bias: old pages that remain live without updates either settled into evergreen niches or previously bottomed out). Staleness is a real directional risk, but it does not act as a monotonic linear decay by itself without factoring in visibility.

---

### Signal Check 2: Search Visibility (`impressions_90d`) behind Opportunity Priority
- **Hypothesis:** Pages with high impression volume have active search demand at stake and register higher measurable decline rates than low-impression pages.
- **Verdict:** **CONFIRMED**
- **Reasoning:** Pages in the lowest impression quintile ($Q_1 \le 39$ impressions) show only a 32.5% decline rate because they are already near the measurement floor. Once pages achieve measurable search presence ($Q_2$ through $Q_4$), decline rates jump above 60% (peaking at 63.3% in $Q_4$). Search visibility separates pages with real traffic to defend from near-zero noise.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Portable repo-root discovery
NB_PATH = Path(os.path.abspath("")).resolve()
REPO_ROOT = NB_PATH
for _ in range(8):
    if (REPO_ROOT / "data" / "raw").is_dir() and (REPO_ROOT / "scripts" / "ml_utils.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError(f"Could not locate repo root starting from: {NB_PATH}")

SCRIPTS_DIR = str(REPO_ROOT / "scripts")
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

RAW_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
assert RAW_PATH.is_file(), f"Missing starter CSV: {RAW_PATH}"

df = pd.read_csv(RAW_PATH)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Loaded {len(df):,} rows from {RAW_PATH.name}")
print(f"Overall dataset decline base rate: {df['is_declining_label'].mean():.3%}\n")

# -------------------------------------------------------------
# Signal 1 (Flag-Linked): Staleness (days_since_last_update)
# -------------------------------------------------------------
print("=" * 65)
print("SIGNAL 1 (Flag-Linked): Staleness / Freshness Tier vs Decline")
print("=" * 65)

table1 = df.groupby("freshness_tier", observed=False).agg(
    n=("content_id", "count"),
    median_days=("days_since_last_update", "median"),
    decline_rate=("is_declining_label", "mean"),
).loc[["0-30", "31-90", "91-180", "181+"]]

table1["decline_pct"] = (table1["decline_rate"] * 100).round(1).astype(str) + "%"
print(table1[["n", "median_days", "decline_pct"]])
print("\nSignal 1 Verdict: MIXED")
print("Takeaway: Decline increases from 51.1% (fresh) to 61.1% (91-180d),")
print("but reverses at 181+ days (47.1%), showing non-linear survivorship bias.\n")

# -------------------------------------------------------------
# Signal 2: Search Visibility (impressions_90d)
# -------------------------------------------------------------
print("=" * 65)
print("SIGNAL 2: Search Visibility (impressions_90d Quintiles) vs Decline")
print("=" * 65)

df["visibility_quintile"] = pd.qcut(
    df["impressions_90d"],
    q=5,
    labels=[
        "Q1 (Low: 1-39)",
        "Q2 (Modest: 40-364)",
        "Q3 (Moderate: 365-1.3k)",
        "Q4 (High: 1.4k-5.1k)",
        "Q5 (Top: 5.2k+)",
    ],
)

table2 = df.groupby("visibility_quintile", observed=False).agg(
    n=("content_id", "count"),
    median_impressions=("impressions_90d", "median"),
    decline_rate=("is_declining_label", "mean"),
)

table2["decline_pct"] = (table2["decline_rate"] * 100).round(1).astype(str) + "%"
print(table2[["n", "median_impressions", "decline_pct"]])
print("\nSignal 2 Verdict: CONFIRMED")
print("Takeaway: Floor effect keeps Q1 decline low (32.5%); once visible,")
print("pages face >60% decline risk, confirming visibility is essential for priority scoring.")


Loaded 30,000 rows from content_refresh_anonymized.csv
Overall dataset decline base rate: 54.207%

SIGNAL 1 (Flag-Linked): Staleness / Freshness Tier vs Decline
                    n  median_days decline_pct
freshness_tier                                
0-30            20480         20.0       51.1%
31-90             175         41.0       58.9%
91-180           9171        104.0       61.1%
181+              174        211.0       47.1%

Signal 1 Verdict: MIXED
Takeaway: Decline increases from 51.1% (fresh) to 61.1% (91-180d),
but reverses at 181+ days (47.1%), showing non-linear survivorship bias.

SIGNAL 2: Search Visibility (impressions_90d Quintiles) vs Decline
                            n  median_impressions decline_pct
visibility_quintile                                          
Q1 (Low: 1-39)           6041                 5.0       32.5%
Q2 (Modest: 40-364)      5964               150.0       60.3%
Q3 (Moderate: 365-1.3k)  5997               732.0       60.5%
Q4 (High: 1.4k

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


### Rule formulation
We construct a transparent, non-fitted baseline score following the live session pattern:
1. **Condition 1 (Staleness):** `days_since_last_update >= 90` (page has not been refreshed in a full quarter).
2. **Condition 2 (Visibility):** `impressions_90d >= 500` (proven organic search demand).
3. **Score:**
   $$\text{baseline\_action\_score} = \mathbb{I}(\text{days} \ge 90) \times \mathbb{I}(\text{imp} \ge 500) \times \text{impressions}_{90d}$$
4. **Reason Code:** `stale_visible_page` (if score > 0) else `insufficient_staleness_or_volume`.
5. **Action Label:** `refresh` (if score > 0) else `monitor`.

Every scored item carries an explicit reason code explaining why it was flagged. No future-window (`*_last_30d`, `*_prev_30d`) or label-derived features (`trend_direction`, `trend_pct`) are used in scoring.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json
from pathlib import Path
import numpy as np
import pandas as pd

# Define output destinations
OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUT = OUTPUT_DIR / "baseline_action_score.csv"
JSON_OUT = OUTPUT_DIR / "baseline_metrics.json"

# 1. Encode the transparent baseline rule
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

# Score scales volume by the binary gate
df["baseline_action_score"] = stale * visible * df["impressions_90d"]

# 2. Attach single reason code and action label
df["reason_code"] = np.where(
    df["baseline_action_score"] > 0,
    "stale_visible_page",
    "insufficient_staleness_or_volume",
)
df["action_label"] = np.where(df["baseline_action_score"] > 0, "refresh", "monitor")

# 3. Build the sorted priority queue
ranked_queue = df.sort_values(
    by=["baseline_action_score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False],
).reset_index(drop=True)
ranked_queue["baseline_rank"] = ranked_queue.index + 1

# 4. Export columns (sanitized, no private client names or raw queries)
output_cols = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "sessions_90d",
    "word_count",
    "content_age_days",
    "is_declining_label",
]

ranked_queue[output_cols].to_csv(CSV_OUT, index=False)
print(f"Exported ranked queue: {CSV_OUT.name} ({len(ranked_queue):,} rows, {CSV_OUT.stat().st_size / 1024:.1f} KB)")


# 5. Precision@K evaluation vs Base Rate
def precision_at_k(queue_df: pd.DataFrame, k: int) -> float:
    return float(queue_df.head(k)["is_declining_label"].mean())


base_rate = float(df["is_declining_label"].mean())
p10 = precision_at_k(ranked_queue, 10)
p20 = precision_at_k(ranked_queue, 20)
p50 = precision_at_k(ranked_queue, 50)

print("\n" + "=" * 55)
print("BASELINE EVALUATION: Precision@K vs Dataset Base Rate")
print("=" * 55)
print(f"Overall Dataset Base Rate : {base_rate:.3%}")
print(f"Precision@10              : {p10:.3%}  ({int(p10*10)} / 10 declining)")
print(f"Precision@20              : {p20:.3%}  ({int(p20*20)} / 20 declining)")
print(f"Precision@50              : {p50:.3%}  ({int(p50*50)} / 50 declining)")

# Save run receipts to JSON (tracked in git)
metrics = {
    "rule_name": "stale_visible_page",
    "score_formula": "stale(days>=90) * visible(imp>=500) * impressions_90d",
    "qualifying_rows": int((df["baseline_action_score"] > 0).sum()),
    "total_rows": int(len(df)),
    "base_rate": round(base_rate, 4),
    "precision_at_10": round(p10, 4),
    "precision_at_20": round(p20, 4),
    "precision_at_50": round(p50, 4),
}

with open(JSON_OUT, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nSaved metrics receipt to {JSON_OUT.name}")

Exported ranked queue: baseline_action_score.csv (30,000 rows, 3386.2 KB)

BASELINE EVALUATION: Precision@K vs Dataset Base Rate
Overall Dataset Base Rate : 54.207%
Precision@10              : 60.000%  (6 / 10 declining)
Precision@20              : 45.000%  (9 / 20 declining)
Precision@50              : 44.000%  (22 / 50 declining)

Saved metrics receipt to baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


Every row in our top ten receives the `refresh` action under the `stale_visible_page` reason code. Here is the individual review of the top 10 candidates:

1. **Rank 1 (`content_5fe46e04994d`):**
   - **Action:** `refresh`
   - **Why it's there:** Peak visibility (517,715 impressions), top-half Page 1 rank (avg position 4.2), 104 days un-updated, and actively declining (`is_declining_label = 1`).
   - **What would make it wrong:** If the decline is due to a structural SERP layout change (e.g. AI Overviews or video carousels occupying the top of page), an editorial text refresh cannot recover lost real estate.

2. **Rank 2 (`content_2dba2b1f9536`):**
   - **Action:** `refresh`
   - **Why it's there:** Huge impression volume (443,434 impressions) and 104 days stale.
   - **What would make it wrong:** It ranks deep on Page 3 (avg position 27.9) and is **not declining** (`is_declining_label = 0`). Prioritizing a Page 3 page over Page 1 decaying assets burns editorial hours for negligible organic traffic lift.

3. **Rank 3 (`content_2c2606c5d176`):**
   - **Action:** `refresh`
   - **Why it's there:** 347,399 impressions, strong Page 1 position (4.2), 2,146 sessions, 104 days stale, and actively declining (`is_declining_label = 1`).
   - **What would make it wrong:** If the decline is driven by competitor domain authority gains or intent shift rather than outdated facts, surface-level content tweaks will fail to reclaim position.

4. **Rank 4 (`content_cb112fce36be`):**
   - **Action:** `refresh`
   - **Why it's there:** 309,910 impressions, Page 1 position (5.6), 104 days stale, and actively declining.
   - **What would make it wrong:** Its CTR is very low (0.16%); if the problem is an unappealing title tag or meta description snippet rather than stale body copy, a full page rewrite is the wrong intervention.

5. **Rank 5 (`content_9532f197bbc8`):**
   - **Action:** `refresh`
   - **Why it's there:** Top ranking (avg position 2.0), 309,192 impressions, 1,098 sessions, 104 days un-updated, and declining.
   - **What would make it wrong:** High intervention risk — aggressively editing an asset holding position #2 risks destabilizing existing keyword relevance signals and triggering further rank loss.

6. **Rank 6 (`content_36ff89c8214e`):**
   - **Action:** `refresh`
   - **Why it's there:** 295,097 impressions, Page 1 position (7.3), 104 days un-updated.
   - **What would make it wrong:** Extremely low CTR (0.05%) and stable/non-declining performance (`is_declining_label = 0`). The page likely ranks on broad secondary queries; editing it will yield little traffic.

7. **Rank 7 (`content_b28d1efd668f`):**
   - **Action:** `refresh`
   - **Why it's there:** 286,608 impressions, 104 days un-updated.
   - **What would make it wrong:** Page 3 ranking (avg position 26.2) with low CTR (0.06%) and not declining (`is_declining_label = 0`). The baseline overweights raw impression scale regardless of ranking tier.

8. **Rank 8 (`content_813e88069237`):**
   - **Action:** `refresh`
   - **Why it's there:** 233,561 impressions, 104 days un-updated, declining (`is_declining_label = 1`).
   - **What would make it wrong:** Another Page 3 position (26.2); while declining, moving a page from rank 26 to 22 produces almost zero clicks compared to defending a page dropping from rank 3 to 7.

9. **Rank 9 (`content_c21024970297`):**
   - **Action:** `refresh`
   - **Why it's there:** Page 1 rank (avg position 5.1), 211,366 impressions, 874 sessions, 104 days un-updated.
   - **What would make it wrong:** It is **not declining** (`is_declining_label = 0`) and holds healthy engagement; unnecessary modification imposes an opportunity cost on editors.

10. **Rank 10 (`content_c8e9d6ab9013`):**
    - **Action:** `refresh`
    - **Why it's there:** 208,678 impressions, position 9.7 (edge of Page 1), 104 days stale, declining.
    - **What would make it wrong:** Near-zero CTR (0.00%) with only 6 sessions over 90 days despite 200k+ impressions. This indicates phantom/bot impressions or broad query matching where real searchers never click.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load exported baseline queue
queue_path = REPO_ROOT / "work" / "outputs" / "baseline_action_score.csv"
queue_df = pd.read_csv(queue_path)

# Display top 10 queue candidates
top10_cols = [
    "baseline_rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "sessions_90d",
    "is_declining_label",
]

top10_df = queue_df.head(10)[top10_cols]
print("=" * 80)
print("TOP 10 BASELINE QUEUE ROWS (FOR SKEPTIC'S REVIEW)")
print("=" * 80)
print(top10_df.to_string(index=False))

print(
    f"\nObserved Decline Rate in Top 10: {top10_df['is_declining_label'].mean():.1%} ({top10_df['is_declining_label'].sum()} / 10)"
)
print(
    f"False Alarm Rate in Top 10    : {(1 - top10_df['is_declining_label'].mean()):.1%} ({10 - top10_df['is_declining_label'].sum()} / 10 non-declining)"
)

TOP 10 BASELINE QUEUE ROWS (FOR SKEPTIC'S REVIEW)
 baseline_rank           content_id  baseline_action_score        reason_code action_label  impressions_90d  days_since_last_update  avg_position  ctr  sessions_90d  is_declining_label
             1 content_5fe46e04994d                 517715 stale_visible_page      refresh           517715                     104           4.2 0.14           520                   1
             2 content_2dba2b1f9536                 443434 stale_visible_page      refresh           443434                     104          27.9 0.21          4218                   0
             3 content_2c2606c5d176                 347399 stale_visible_page      refresh           347399                     104           4.2 0.53          2146                   1
             4 content_cb112fce36be                 309910 stale_visible_page      refresh           309910                     104           5.6 0.16           480                   1
             5 content_95

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



### 1. Which picks look wrong and why?

A critical review of the top 20 reveals three clear systematic failure modes in our baseline rule:

1. **Deep-ranking (Page 3) False Alarms (e.g. Rank 2 `content_2dba2b1f9536` and Rank 7 `content_b28d1efd668f`):**
   - **Why they look wrong:** Both pages rank past position 26 (deep Page 3). Because our baseline score multiplies directly by raw `impressions_90d`, massive head-term queries showing on Page 3 get pushed to the top of the queue despite having tiny click-through rates. Furthermore, neither page is actually declining (`is_declining_label = 0`). Directing editors to refresh deep Page 3 pages wastes scarce human hours.
   - **Fix for the Week 5 ML model:** The model should incorporate `avg_position` and position tiers so it penalizes or discounts pages ranking beyond Page 1/2 striking distance.

2. **Phantom / Zero-Click Impressions (Rank 10 `content_c8e9d6ab9013`):**
   - **Why it looks wrong:** This page accumulated 208,678 impressions, yet its CTR is 0.00% and it generated only 6 sessions over 90 days. These impressions are likely bot activity or obscure image carousels rather than human searchers reading text.
   - **Fix for the Week 5 ML model:** The model should require minimum session or click thresholds (`has_clicks`, `sessions_90d >= 10`), avoiding phantom-impression traps.

3. **Stable / Non-Declining Assets (Ranks 6, 9, 12, 13, 14, 17, 18, 20):**
   - **Why they look wrong:** The baseline flags them solely because `days_since_last_update >= 90`, but their organic traffic is healthy or growing (`is_declining_label = 0`). Overwriting stable, well-ranking pages introduces unnecessary risk of keyword destabilization.

---

### 2. Leakage check confirmation

We conducted a strict audit against target leakage and future-window contamination:
- **Zero Label Contamination:** The rule does not use `is_declining_label`, nor does it use the label's parent fields `trend_direction` or `trend_pct`.
- **Zero Future-Window Leakage:** The rule uses strictly historical trailing-90-day snapshot features (`days_since_last_update`, `impressions_90d`). It does not use any sub-window split metrics (`*_last_30d` or `*_prev_30d`).
- **No Product Flags:** The score is built directly from raw numeric columns without relying on pre-computed product flags.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load the ranked baseline queue
queue_path = REPO_ROOT / "work" / "outputs" / "baseline_action_score.csv"
queue_df = pd.read_csv(queue_path)

print("=" * 75)
print("WEAK PICKS AUDIT IN TOP 20")
print("=" * 75)

# 1. Flag weak picks: Deep ranking positions (> 20) with low click yield
deep_ranks = queue_df.head(20)[queue_df.head(20)["avg_position"] > 20]
print(f"1. Deep Positions (Avg Position > 20) in Top 20: {len(deep_ranks)} pages")
print(
    deep_ranks[
        [
            "baseline_rank",
            "content_id",
            "avg_position",
            "impressions_90d",
            "ctr",
            "is_declining_label",
        ]
    ].to_string(index=False)
)

# 2. Flag weak picks: Phantom impressions (CTR < 0.05% with sessions < 10)
phantom = queue_df.head(20)[
    (queue_df.head(20)["ctr"] < 0.05) & (queue_df.head(20)["sessions_90d"] < 10)
]
print(f"\n2. Phantom / Zero-Click Impressions in Top 20: {len(phantom)} page(s)")
if len(phantom) > 0:
    print(
        phantom[
            [
                "baseline_rank",
                "content_id",
                "impressions_90d",
                "ctr",
                "sessions_90d",
                "is_declining_label",
            ]
        ].to_string(index=False)
    )

# -------------------------------------------------------------
# Programmatic Leakage Assertions
# -------------------------------------------------------------
print("\n" + "=" * 75)
print("LEAKAGE CHECK VERIFICATION")
print("=" * 75)

forbidden_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

# The rule strictly depends on these two columns:
rule_feature_inputs = ["days_since_last_update", "impressions_90d"]

overlap = [col for col in rule_feature_inputs if col in forbidden_columns]
assert len(overlap) == 0, f"Leakage detected! Rule uses forbidden columns: {overlap}"

print("Features used in baseline rule :", rule_feature_inputs)
print("Forbidden columns audited      :", forbidden_columns)
print("Forbidden column overlap       : 0 (PASSED)")
print("Status                         : CLEAN — No label or future-window leakage.")

WEAK PICKS AUDIT IN TOP 20
1. Deep Positions (Avg Position > 20) in Top 20: 6 pages
 baseline_rank           content_id  avg_position  impressions_90d  ctr  is_declining_label
             2 content_2dba2b1f9536          27.9           443434 0.21                   0
             7 content_b28d1efd668f          26.2           286608 0.06                   0
             8 content_813e88069237          26.2           233561 0.06                   1
            11 content_b511d4bc4ad2          27.9           205915 0.14                   0
            18 content_f02b48f88241          25.8           181514 0.10                   0
            19 content_05e9b4cd9ccf          22.1           179002 0.08                   1

2. Phantom / Zero-Click Impressions in Top 20: 1 page(s)
 baseline_rank           content_id  impressions_90d  ctr  sessions_90d  is_declining_label
            10 content_c8e9d6ab9013           208678  0.0             6                   1

LEAKAGE CHECK VERIFICATION
Fe

Note:
A baseline's goal is to simulate what an editorial team or SEO director builds by hand in a spreadsheet with no ML background.
They think in raw numbers: "Show me the stale pages with the most impressions first."
Raw impressions keep the score intuitive and completely transparent without any math transformations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.